## PER2526, grupo de laboratorio L1-3CO11
## A3. Prueba práctica de laboratorio de B2 (cuaderno) (1.25 puntos)
## 26 de mayo de 2026

**Ejercicio:** $\;$ Suponiendo que queremos adaptar el modelo "resnet50.fb_swsl_ig1b_ft_in1k" a la tarea Cifar-10 siguiendo los pasos realizados en las sesiones de prácticas, completa el siguiente cuaderno para preparar un experimento en que se realize un finetuning. En el experimento, entrenaremos el modelo durante 5 épocas utilizando una transformación de redimensionado con tamaño `(18,)`, además de un scheduler `CosineAnnealing` con un learning rate minimo de `1e-6`.

**Nota:** $\;$ Antes de empezar el experimento, debes introducir las **últimas 3** cifras de tu DNI/NIE en el campo `dni` de la segunda celda del cuaderno

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import timm
from torchvision.transforms import v2
from torch.utils.data import DataLoader
import datasets

In [2]:
dni=673 # Completar

In [3]:
def exp(model, device, train_loader, test_loader, optimizer, scheduler, epochs=15):
    loss_fn = torch.nn.CrossEntropyLoss()
    for epoch in range(epochs):
        print(f'Epoch {epoch}: train:', end=' ')
        model.train(); trsize, trbatches, trloss, tracc = 0, 0, 0, 0
        for batch in train_loader:
            X, y = batch['X'].to(device), batch['y'].to(device); trsize += len(X); trbatches += 1
            pred = model(X); loss = loss_fn(pred, y); trloss += loss.item()
            tracc += (pred.argmax(1) == y).type(torch.float).sum().item()
            loss.backward(); optimizer.step(); optimizer.zero_grad(); scheduler.step()
        trloss /= trbatches; tracc /= trsize
        print(f'loss {trloss:g} acc {tracc:.2%} test:', end=' ')
        model.eval(); tesize, tebatches, teloss, teacc = 0, 0, 0, 0
        with torch.no_grad():
            for batch in test_loader:
                X, y = batch['X'].to(device), batch['y'].to(device); tesize += len(X); tebatches += 1
                pred = model(X); teloss += loss_fn(pred, y).item()
                teacc += (pred.argmax(1) == y).type(torch.float).sum().item()
        teloss /= tebatches; teacc /= tesize
        print(f'loss {teloss:g} acc {teacc:.2%}')

In [4]:
ds = datasets.load_dataset("uoft-cs/cifar10").rename_columns({'img': 'X', 'label': 'y'})
train_ds = ds['train'].to_iterable_dataset(num_shards=1024).shuffle(seed=dni)
test_ds = ds['test'].to_iterable_dataset(num_shards=1024)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/23.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [5]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
model_name = "resnet50.fb_swsl_ig1b_ft_in1k"
model = timm.create_model(model_name, pretrained=True, num_classes = 10).to(device) # Completar

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

In [6]:
data_cfg = timm.data.resolve_data_config(model.pretrained_cfg)
data_cfg['input_size'] = (3,32,32) # Completar
timm.data.create_transform(**data_cfg)

Compose(
    Resize(size=36, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(32, 32))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)

In [7]:
train_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.uint8, scale=True),
    # v2.RandomHorizontalFlip(),
    v2.Resize(size=(18,), max_size=None),
    # v2.CenterCrop(size=(32, 32)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=data_cfg["mean"], std=data_cfg["std"]),
    ]) # Completar
def train_prep(example):
    return { 'X' : train_transform(example['X']), 'y' : example['y']}
train_ds = train_ds.map(train_prep, batched=True)
train_loader = DataLoader(train_ds, batch_size=32)

test_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=data_cfg["mean"], std=data_cfg["std"]),
    ]) # Completar
def test_prep(example):
    return { 'X' : test_transform(example['X']), 'y' : example['y']}
test_ds = test_ds.map(test_prep, batched=True)
test_loader = DataLoader(test_ds, batch_size=32)

In [8]:
for p in model.parameters():
    p.requires_grad = False

for p in model.fc.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
steps = 5
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, steps, eta_min=1e-6) # Completar

exp(model, device, train_loader, test_loader, optimizer, scheduler, epochs=steps)

Epoch 0: train: loss 1.90809 acc 34.29% test: loss 3.1167 acc 32.52%
Epoch 1: train: loss 1.72475 acc 40.65% test: loss 3.61186 acc 31.43%
Epoch 2: train: loss 1.70695 acc 41.39% test: loss 3.89781 acc 30.73%
Epoch 3: train: loss 1.70368 acc 41.75% test: loss 4.00133 acc 30.34%
Epoch 4: train: loss 1.70258 acc 41.98% test: loss 4.10956 acc 30.07%



<p style="page-break-after:always;"></p>
